# 3주차 과제 베이스라인 — 전력사용량 예측

100개 건물의 2022년 여름 시간별 전력사용량 데이터입니다. 원본은 README에 안내된 [전력사용량 예측 AI 경진대회](https://dacon.io/competitions/official/236125) 페이지에서 내려받을 수 있습니다. 내려받은 `train.csv`와 `building_info.csv`를 `dataset/open/extracted/2023 전력사용량 예측 AI 경진대회/`에 두면 아래 로딩 코드와 연결됩니다. 이 노트북은 건물별 기상 관측치와 건물 특성 정보를 조인하고 결측치 처리 예시를 하나 보여 줍니다. **나머지 결측치는 발생 이유를 확인하고 처리 근거를 설명하는 과정이 이번 과제의 핵심입니다.**

이번 과제에서는 높은 성능보다 **직접 시도한 코드, 만난 오류, 문제를 더 작게 나눈 과정, 다음 행동**을 중요하게 기록합니다.

- ✅ **기본 시도**: 파일 확인, 조인, 공통 기준 시각 분할, 작은 트리 실행까지 진행합니다.
- 🧩 **문제 분해**: 결측 변수 하나 또는 과적합 하나를 골라 확인합니다.
- 🌱 **선택 탐색**: 결측 처리와 가지치기 설정을 비교합니다.


## 1. ✅ 기본 시도 — 데이터 불러오기와 조인


In [ ]:
from pathlib import Path

import pandas as pd

start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [start, *start.parents] if (p / '03주차' / 'assignment_baseline.ipynb').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('README.md와 03주차 폴더가 있는 프로젝트 안에서 노트북을 실행합니다.')

data_dir = PROJECT_ROOT / 'dataset' / 'open' / 'extracted' / '2023 전력사용량 예측 AI 경진대회'
train_path = data_dir / 'train.csv'
building_path = data_dir / 'building_info.csv'
missing_files = [p.name for p in [train_path, building_path] if not p.exists()]
print('확인한 데이터 폴더:', data_dir)
print('누락 파일:', missing_files if missing_files else '없음')

if not missing_files:
    DATA_MODE = '실제 데이터'
    train = pd.read_csv(train_path)
    building = pd.read_csv(building_path)
else:
    DATA_MODE = '연습용 소규모 예시 데이터'
    demo_rows = []
    for building_no in [1, 2]:
        for ts in pd.date_range('2022-06-01', periods=48, freq='h'):
            hour = ts.hour
            demo_rows.append({
                'num_date_time': f'{building_no}_{ts:%Y%m%d %H}',
                '건물번호': building_no,
                '일시': f'{ts:%Y%m%d %H}',
                '기온(C)': 18 + hour * 0.4,
                '강수량(mm)': 2.0 if hour == 15 else None,
                '풍속(m/s)': None if hour == 3 else 1.2 + hour * 0.03,
                '습도(%)': 65 - hour * 0.5,
                '일조(hr)': 1.0 if 7 <= hour <= 18 else None,
                '일사(MJ/m2)': 0.8 if 7 <= hour <= 18 else None,
                '전력소비량(kWh)': 800 + building_no * 100 + hour * 15 + (150 if 12 <= hour <= 18 else 0),
            })
    train = pd.DataFrame(demo_rows)
    building = pd.DataFrame({
        '건물번호': [1, 2],
        '건물유형': ['연습용 사무실', '연습용 상가'],
        '연면적(m2)': [10000.0, 15000.0],
        '냉방면적(m2)': [6000.0, 9000.0],
        '태양광용량(kW)': ['-', '30.0'],
        'ESS저장용량(kWh)': ['-', '-'],
        'PCS용량(kW)': ['-', '-'],
    })
    print('실제 파일을 준비하기 전까지 예시 데이터로 실행 흐름만 연습합니다.')

print('데이터 모드:', DATA_MODE)
print('train/building 크기:', train.shape, building.shape)


`building_info`의 `태양광용량`, `ESS저장용량`, `PCS용량`은 해당 설비가 없는 건물에 결측치(NaN)가 아니라 문자열 `'-'`로 표시되어 있습니다. 데이터 명세에 따라 이를 '설비 없음 = 0'으로 해석하고, 수치 계산이 가능하도록 숫자로 변환합니다.


In [ ]:
equip_cols = ['태양광용량(kW)', 'ESS저장용량(kWh)', 'PCS용량(kW)']
for c in equip_cols:
    building[c] = building[c].replace('-', 0).astype(float)

building.head()


In [ ]:
df = train.merge(building, on='건물번호', how='left')
df.shape


## 2. 🧩 문제 분해 — 결측치 확인


In [ ]:
df.isna().mean().sort_values(ascending=False)


결측 비율은 컬럼마다 크게 다릅니다. `강수량`은 약 78%가 비어 있으며, 무강수 시간에 값을 기록하지 않은 방식일 가능성도 하나의 가설입니다. 원자료 설명, 다른 기상 변수, 시간대별 분포를 함께 살펴 이 가설을 확인할 수 있습니다. 아래 셀은 0 대치의 코드 예시를 보여 주며, 실제 분석에서는 확인 결과에 따라 다른 처리 방법과 비교합니다.


## 3. 🧩 문제 분해 — 강수량 처리 예시

**아래는 강수량 처리의 한 가지 예시입니다. `풍속`, `습도`, `일조`, `일사`는 변수의 의미와 결측 발생 과정이 다를 수 있으므로, 각 컬럼의 시간대·건물별 결측 패턴을 확인한 뒤 처리 방법을 선택합니다.**


In [ ]:
df['강수량(mm)'] = df['강수량(mm)'].fillna(0)
df['강수량(mm)'].isna().sum()


## 이제 분석을 이어서 작성합니다

먼저 ✅ 표시가 있는 공통 시각 분할과 작은 트리를 실행합니다. 다른 결측 열을 모두 처리하거나 설정을 많이 비교하는 일은 기본 기록을 남긴 뒤 진행해도 됩니다.


### 4. 🧩 문제 분해 — 나머지 결측치 진단

힌트: `일시` 컬럼에서 시간(hour)을 추출해 낮과 밤의 결측 비율을 비교합니다. 예를 들어 `일조(hr)`가 밤 시간대에 주로 비어 있다면 '관측 실패'뿐 아니라 '일조가 적용되지 않는 시간'일 가능성도 살펴봅니다. 의미가 있는 구조적 빈칸과 관측 실패로 생긴 결측은 구분한 뒤 각각에 맞는 처리 방법을 선택합니다.


In [ ]:
df['일시_dt'] = pd.to_datetime(df['일시'], format='%Y%m%d %H', errors='coerce')
df['시간'] = df['일시_dt'].dt.hour
check_cols = ['풍속(m/s)', '습도(%)', '일조(hr)', '일사(MJ/m2)']
hourly_missing = df.groupby('시간')[check_cols].agg(lambda s: s.isna().mean())
display(hourly_missing.round(3))
print('작은 힌트: 낮과 밤의 결측률이 다른 열 하나만 골라 가능한 수집 이유를 적습니다.')


### 5. ✅ 기본 시도 — 변수 선택 및 공통 시각 분할

예측 대상은 `전력소비량(kWh)`입니다. 예측 시점에 사용할 수 있는 변수를 고르고, `건물번호`와 `건물유형` 같은 범주형 변수에는 적절한 인코딩을 적용합니다. 시간 순서를 고려해 학습·검증 데이터를 나누면 미래 예측 상황에 가까운 평가를 만들 수 있습니다.


In [ ]:
df['일시_dt'] = pd.to_datetime(df['일시'], format='%Y%m%d %H', errors='coerce')
df['시간'] = df['일시_dt'].dt.hour

df = df.dropna(subset=['일시_dt']).sort_values(['일시_dt', '건물번호']).reset_index(drop=True)
df['요일번호'] = df['일시_dt'].dt.dayofweek
basic_features = ['기온(C)', '시간', '요일번호', '연면적(m2)']
target = '전력소비량(kWh)'
model_df = df.dropna(subset=basic_features + [target]).copy()

unique_times = pd.Index(model_df['일시_dt'].drop_duplicates().sort_values())
split_position = min(max(int(len(unique_times) * 0.8), 1), len(unique_times) - 1)
validation_start = pd.Timestamp(unique_times[split_position])
fit_df = model_df[model_df['일시_dt'] < validation_start]
valid_df = model_df[model_df['일시_dt'] >= validation_start]

X_train, y_train = fit_df[basic_features], fit_df[target]
X_valid, y_valid = valid_df[basic_features], valid_df[target]
print('검증 시작 시각:', validation_start)
print('학습/검증 행 수:', len(fit_df), len(valid_df))
print('학습 마지막 < 검증 시작:', fit_df['일시_dt'].max() < valid_df['일시_dt'].min())
print('기본 경로에서는 결측이 없는 숫자 변수 네 개만 사용합니다.')


### 6. ✅ 기본 시도 — 작은 트리 한 번 실행

먼저 깊이와 잎 크기를 제한한 작은 트리 하나만 실행합니다. 아래 시작값은 정답이 아니며, 첫 결과나 오류를 얻어 다음 질문을 정하기 위한 값입니다.


In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor

def rmse(actual, predicted):
    return mean_squared_error(actual, predicted) ** 0.5

first_leaf_size = max(2, min(50, len(X_train) // 20))
first_tree = DecisionTreeRegressor(
    max_depth=6, min_samples_leaf=first_leaf_size, random_state=42
)
first_tree.fit(X_train, y_train)
first_result = pd.DataFrame({
    'train RMSE': [rmse(y_train, first_tree.predict(X_train))],
    'validation RMSE': [rmse(y_valid, first_tree.predict(X_valid))],
}, index=['트리(작은 첫 시도)']).round(2)
display(first_result)
print('사용한 min_samples_leaf:', first_leaf_size)
print('이 validation 결과를 최종 test 점수라고 부르지 않습니다.')


### 7. 🌱 선택 탐색 — 선형회귀와 제한 없는 트리 비교

기본 기록을 남긴 뒤 아래 셀을 실행합니다. 같은 분할에서 선형회귀와 제한 없는 트리를 비교하고, train/validation 격차를 살펴봅니다.


In [ ]:
from sklearn.linear_model import LinearRegression

extra_models = {
    '선형회귀': LinearRegression(),
    '트리(제한없음)': DecisionTreeRegressor(random_state=42),
}
rows = [first_result.reset_index().rename(columns={'index': '모델'}).iloc[0].to_dict()]
for name, model in extra_models.items():
    model.fit(X_train, y_train)
    rows.append({
        '모델': name,
        'train RMSE': rmse(y_train, model.predict(X_train)),
        'validation RMSE': rmse(y_valid, model.predict(X_valid)),
    })
results = pd.DataFrame(rows).set_index('모델').round(2)
display(results)


### 8. 이번 주 학습 기록

실행 성공보다 시도·오류·문제 분해 기록을 우선합니다. 아래 네 줄을 자신의 말로 채웁니다.

- 질문 1개: 예) 기온·시간·건물 정보로 이후 전력사용량을 어느 정도 예측할 수 있습니까?
- 실행 또는 실행 시도 1개: 실행한 셀과 데이터 모드
- 관찰 결과 또는 오류 1개: RMSE 차이, 결측 패턴, 누락 파일명 또는 오류 메시지
- 다음 행동 1개: 결측 열 하나 확인, 경로 수정, 가지치기 후보 비교 등

작은 힌트: 오류가 나면 `파일 → 열 이름 → 조인 행 수 → 날짜 변환 → 분할 행 수 → X의 결측` 순서로 확인합니다.

> **여기까지 하면 이번 주 기록 완료입니다.** 모든 결측 열 처리와 하이퍼파라미터 탐색은 선택 탐색입니다.
